In [1]:
import kagglehub
import numpy as np
import os
import time
import multiprocessing as mp
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.decomposition import PCA

In [2]:
# Descargar el dataset
dataset_ruta_kaggle = kagglehub.dataset_download(
    "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
)

Using Colab cache for faster access to the 'gtsrb-german-traffic-sign' dataset.


In [3]:
# Cargar datos
datos = []
etiquetas = []
clases = 43

IMAGES_PER_CLASS = 300

for i in range(clases):
    ruta = os.path.join(dataset_ruta_kaggle, 'train', str(i))
    images = os.listdir(ruta)[:IMAGES_PER_CLASS]

    for a in images:
        try:
            img = Image.open(os.path.join(ruta, a)).convert('RGB') # Convertir a RGB
            img = img.resize((30, 30)) # Redimensionar a 30x30
            img = np.array(img, dtype=np.uint8) # Convertir a array numpy
            datos.append(img) # Agregar imagen a datos
            etiquetas.append(i) # Agregar etiqueta a etiquetas
        except:
            pass

datos = np.array(datos)
etiquetas = np.array(etiquetas)

print("Dataset cargado:", datos.shape)

Dataset cargado: (12330, 30, 30, 3)


In [4]:
# División del dataset
X_train, X_test, y_train, y_test = train_test_split(
    datos, etiquetas, test_size=0.2, random_state=42)

In [5]:
X_train = X_train.reshape(X_train.shape[0], -1).astype(np.float32) / 255.0 # Normalizar a [0, 1]
X_test  = X_test.reshape(X_test.shape[0], -1).astype(np.float32) / 255.0 # Normalizar a [0, 1]

pca = PCA(n_components=100) # Reducir a 100 componentes principales
X_train = pca.fit_transform(X_train) # Ajustar PCA solo con el conjunto de entrenamiento
X_test  = pca.transform(X_test) # Transformar el conjunto de prueba con los componentes principales ajustados

print("Dimensiones después de PCA:", X_train.shape)

Dimensiones después de PCA: (9864, 100)


In [6]:
# SVM
def evaluar_svm():
    model = LinearSVC()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    return acc, pred

In [7]:
# Parte Secuencial
start_seq = time.time()
acc_seq, pred_seq = evaluar_svm()
t_seq = time.time() - start_seq

In [8]:
# Paralelo
def train_subset(data):
    X_sub, y_sub = data
    model = LinearSVC()
    model.fit(X_sub, y_sub)
    return model

NUM_PROCESOS = 4

# dividir dataset
indices = np.array_split(np.arange(len(X_train)), NUM_PROCESOS)
data_splits = [(X_train[idx], y_train[idx]) for idx in indices]

start_par = time.time()

with mp.Pool(processes=NUM_PROCESOS) as pool:
    models = pool.map(train_subset, data_splits)

t_par = time.time() - start_par

# usar un modelo para evaluar (simple)
pred_par = models[0].predict(X_test)
acc_par = accuracy_score(y_test, pred_par)

In [9]:
# Métricas
speedup = t_seq / t_par if t_par > 0 else float('inf')
eficiencia = (speedup / NUM_PROCESOS) * 100

In [10]:
print("\n===== PARALELIZACIÓN =====")
print(f"Tiempo secuencial: {t_seq:.2f} seg")
print(f"Tiempo paralelo: {t_par:.2f} seg")
print(f"Speedup: {speedup:.2f}")
print(f"Eficiencia: {eficiencia:.2f}%")

print("\n===== MÉTRICAS MODELO =====")
print(f"Exactitud: {acc_seq:.4f}")
print(f"Precision: {precision_score(y_test, pred_seq, average='macro'):.4f}")
print(f"Recall: {recall_score(y_test, pred_seq, average='macro'):.4f}")
print(f"F1-score: {f1_score(y_test, pred_seq, average='macro'):.4f}")


===== PARALELIZACIÓN =====
Tiempo secuencial: 9.86 seg
Tiempo paralelo: 7.49 seg
Speedup: 1.32
Eficiencia: 32.94%

===== MÉTRICAS MODELO =====
Exactitud: 0.8560
Precision: 0.8604
Recall: 0.8609
F1-score: 0.8580
